In [ ]:
CSV_PATH = "P1_S12_EMG1kHz.txt"
META_PATH = "P1_S12.json"
CHANNELS_PATH = "P2_metadata.json"
TASKS_PATH = "movements.json"

# 1. Trial-level metadata (already one row per trial)
trial_meta = pd.read_json(META_PATH)

# 2. Reshape EMG csv: group by trialID -> (numSamples, numChannels) array per trial
emg_raw = pd.read_csv(CSV_PATH)
channel_cols = [c for c in emg_raw.columns if c.startswith('EMG1k_')]

emg_arrays = {
    tid: group[channel_cols].to_numpy()
    for tid, group in emg_raw.groupby('trialID')
}
trial_meta['EMG1k'] = trial_meta['TrialID'].map(emg_arrays)

# sanity check
print(trial_meta['EMG1k'].iloc[0].shape)  # (8000, 8) confirmed

# 3. Task names
tasks = pd.read_json(TASKS_PATH)
tasks = tasks.rename(columns={'movementNumber': 'TaskNumber', 'movementName': 'TaskName'})
tasks['TaskNumber'] = tasks['TaskNumber'].astype(int)
trial_meta['TaskNumber'] = trial_meta['TaskNumber'].astype(int)
trial_meta = trial_meta.merge(tasks[['TaskNumber', 'TaskName']], on='TaskNumber', how='left')

# 4. Channel names -- lookup dict, not merged row-wise (constant across all trials)
channels = pd.read_json(CHANNELS_PATH)
channel_names = channels.set_index('channelNumber')['channelName'].to_dict()
# {1: 'FDPI', 2: 'FCR', 3: 'Ulnar RPNI', 4: 'Median RPNI', 5: 'EDC', 6: 'EPL', 7: 'FDPS', 8: 'FPL'}

trial_meta.head()